In [1]:
!pip install -q -U transformers accelerate requests

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import glob
import time
import zlib
import requests
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 40.7 MB/s eta 0:00:00
device: cuda


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

layers = model.model.language_model.layers
n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
IMAGE_TOKEN_ID = model.config.image_token_id
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

28 language model layers, 28 heads, 128 head dim, image token id 151655


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_CACHE_DIR = "/kaggle/temp/vg"
os.makedirs(VG_CACHE_DIR, exist_ok=True)
VG_URLS = ["https://cs.stanford.edu/people/rak248/VG_100K", "https://cs.stanford.edu/people/rak248/VG_100K_2"]

NOISE_LEVELS = [3.0]
GRAD_SCALES = [32.0, 1.0]
GRAD_SCALE = GRAD_SCALES[0]
TAIL_N = 5
GROUPS = ["visual", "text", "tail"]
BATCH_SAVE_SIZE = 200
OUT_DIR = "/kaggle/working/atp_qwen"
os.makedirs(OUT_DIR, exist_ok=True)


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id):
    local = os.path.join(VG_CACHE_DIR, f"{image_id}.jpg")
    if os.path.exists(local):
        return local
    for attempt in range(3):
        for base in VG_URLS:
            try:
                r = requests.get(f"{base}/{image_id}.jpg", timeout=30)
            except requests.RequestException:
                continue
            if r.status_code == 200 and len(r.content) > 1000:
                with open(local + ".part", "wb") as f:
                    f.write(r.content)
                os.replace(local + ".part", local)
                return local
        time.sleep(2 * (attempt + 1))
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/dla-qwen/dla_meta.json") as f:
    dla_meta = json.load(f)
PAIRS = {k: tuple(v) for k, v in dla_meta["pairs"].items()}
for prefix, (yes_id, no_id) in PAIRS.items():
    words = [processor.tokenizer.decode([i]).strip() for i in (yes_id, no_id)]
    print(prefix, yes_id, no_id, words)
    assert [w.lower() for w in words] == ["yes", "no"], words
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)

with open(f"{INPUT}/atp-qwen-selection/selection.json") as f:
    selection = json.load(f)
all_sel = [qid for t in selection["selected"] for qid in selection["selected"][t]]
assert len(all_sel) == len(set(all_sel)) == 2927, "attach atp-qwen-selection version 1"
print(len(all_sel), "selected questions")


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == int(inputs["image_grid_thw"].prod()) // 4 and not bool((vis & tail).any())
    yes_id, no_id = PAIRS[qid.split("_")[0] + "_"]
    return {
        "qid": qid, "T": T, "inputs": inputs, "vis": vis, "yes_id": yes_id, "no_id": no_id,
        "masks": {"visual": vis.float().cpu(), "text": text.float().cpu(), "tail": tail.float().cpu()},
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }

amber_ 9454 2753 ['Yes', 'No']
reefknot_ 9693 2152 ['yes', 'no']
2927 selected questions


In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "grad": False, "live": None, "vis": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0:
            if STATE["noise"] is not None:
                vis = STATE["vis"]
                h = h.clone()
                x = h[:, vis].float()
                generator = torch.Generator().manual_seed(STATE["seed"])
                noise = torch.randn(x.shape, generator=generator).to(h.device)
                h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
                changed = True
            if STATE["grad"]:
                h = h.detach().requires_grad_(True)
                changed = True
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        if STATE["capture"]:
            STATE["live"]["z"][l] = args[0]
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = output
    return hook


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
print(len(_handles), "hooks registered")

84 hooks registered


In [5]:
def mask_on(p, gname, dev):
    cache = p.setdefault("_mask", {})
    key = (gname, str(dev))
    if key not in cache:
        cache[key] = p["masks"][gname].to(dev)
    return cache[key]


def read_delta(p, out):
    logits = out.logits[0, -1].float()
    return p["sign"] * (logits[p["yes_id"]] - logits[p["no_id"]])


def capture_run(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, grad=True, vis=p["vis"])
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    with torch.enable_grad():
        out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
        delta = read_delta(p, out)
        live = STATE["live"]
        grads = torch.autograd.grad(delta * GRAD_SCALE, live["z"] + live["mlp"] + live["rp"])
    finite = all(bool(torch.isfinite(g).all()) for g in grads)
    acts = {k: [t.detach() for t in live[k]] for k in ("z", "mlp", "rp")}
    G = {"z": list(grads[:n_layer]), "mlp": list(grads[n_layer:2 * n_layer]), "rp": list(grads[2 * n_layer:])}
    result = {"acts": acts, "grads": G, "delta": float(delta.detach()), "finite": finite}
    STATE.update(capture=False, grad=False, live=None, noise=None)
    del out, delta, live, grads
    return result


@torch.no_grad()
def capture_forward(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, grad=False, vis=p["vis"])
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
    delta = read_delta(p, out)
    result = {"acts": STATE["live"], "delta": float(delta)}
    STATE.update(capture=False, live=None, noise=None)
    del out
    return result


def atp_effects(A, B, G, p):
    head = torch.zeros(len(GROUPS), n_layer, n_head)
    mlp = torch.zeros(len(GROUPS), n_layer)
    rp = torch.zeros(len(GROUPS), n_layer)
    T = p["T"]
    for l in range(n_layer):
        dz = ((A["z"][l].float() - B["z"][l].float()) * G["z"][l].float() / GRAD_SCALE)[0].view(T, n_head, head_dim).sum(-1)
        dm = ((A["mlp"][l].float() - B["mlp"][l].float()) * G["mlp"][l].float() / GRAD_SCALE)[0].sum(-1)
        dr = ((A["rp"][l].float() - B["rp"][l].float()) * G["rp"][l].float() / GRAD_SCALE)[0].sum(-1)
        for gi, g in enumerate(GROUPS):
            head[gi, l] = (mask_on(p, g, dz.device) @ dz).cpu()
            mlp[gi, l] = (mask_on(p, g, dm.device) @ dm).cpu()
            rp[gi, l] = (mask_on(p, g, dr.device) @ dr).cpu()
    return {"head": head, "mlp": mlp, "rp": rp}


def process_once(p, seed):
    global GRAD_SCALE
    clean, scale_used = None, None
    for scale in GRAD_SCALES:
        GRAD_SCALE = scale
        c = capture_run(p, None, seed)
        if c["finite"] and np.isfinite(c["delta"]):
            clean, scale_used = c, scale
            break
        del c
    if clean is None:
        GRAD_SCALE = GRAD_SCALES[0]
        return None, "clean_nonfinite"
    result, reason = None, "corrupt_nonfinite"
    for noise in NOISE_LEVELS:
        r = capture_forward(p, noise, seed)
        if np.isfinite(r["delta"]):
            eff = atp_effects(r["acts"], clean["acts"], clean["grads"], p)
            if all(bool(torch.isfinite(eff[k]).all()) for k in eff):
                result = {
                    "delta_clean": clean["delta"], "delta_corrupt": r["delta"], "TE": clean["delta"] - r["delta"],
                    "noise_used": noise, "grad_scale_used": scale_used,
                    "head": eff["head"], "mlp": eff["mlp"], "rp": eff["rp"],
                }
                del r
                break
        del r
    GRAD_SCALE = GRAD_SCALES[0]
    del clean
    return result, reason


def process(p, seed):
    for attempt in range(2):
        try:
            return process_once(p, seed)
        except torch.cuda.OutOfMemoryError:
            STATE.update(capture=False, grad=False, live=None, noise=None)
            torch.cuda.empty_cache()
    return None, "oom"

In [6]:
def collect(tag, keep):
    todo_ids = [qid for qid in all_sel if keep(qid)]
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{tag}_batch_*.pt")))
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [qid for qid in todo_ids if qid not in done_ids]
    print(f"{tag}: {len(todo_ids)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, missing, failed = [], len(paths), 0, 0
    t_start = time.time()

    for i, qid in enumerate(todo):
        q = question_by_id[qid]
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            missing += 1
            continue

        p = prep(qid)
        seed = zlib.crc32(qid.encode()) % (2 ** 31)
        result, reason = process(p, seed)
        entry = {
            "question_id": qid, "image_id": q["image_id"], "hallucination_type": q["hallucination_type"],
            "ground_truth_answer": q["ground_truth_answer"], "stratum": selection["stratum"][qid],
            "group": selection["group"][qid], "source": selection["source"][qid], "split": selection["split"][qid],
            "sign": p["sign"], "seed": seed, "ok": result is not None, "T": p["T"], "n_visual": int(p["vis"].sum()),
        }
        if result is None:
            entry["reason"] = reason
            failed += 1
        else:
            entry.update(result)
            entry["pred_yes"] = bool(p["sign"] * result["delta_clean"] > 0)
        batch_entries.append(entry)
        del p

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1

        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
            elapsed = time.time() - t_start
            avg = elapsed / (i + 1)
            print(f"[{i + 1}/{len(todo)}] {qid} ({avg:.2f} s/question avg, ETA {avg * (len(todo) - i - 1) / 60:.1f} min, "
                  f"failed {failed}, missing images {missing})")

    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
    print(f"done -- {tag}, failed {failed}, missing images {missing}, {(time.time() - t_start) / 60:.1f} min")
    return missing

In [7]:
missing = collect("main", lambda qid: not qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

main: 754 questions, 0 already done, 754 to run


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[101/754] amber_8218 (1.96 s/question avg, ETA 21.3 min, failed 0, missing images 0)
[201/754] amber_1444 (1.97 s/question avg, ETA 18.2 min, failed 0, missing images 0)
[301/754] amber_8519 (2.01 s/question avg, ETA 15.2 min, failed 0, missing images 0)
[401/754] amber_4363 (2.03 s/question avg, ETA 11.9 min, failed 0, missing images 0)
[501/754] amber_14658 (2.03 s/question avg, ETA 8.6 min, failed 0, missing images 0)
[601/754] amber_14828 (2.05 s/question avg, ETA 5.2 min, failed 0, missing images 0)
[701/754] amber_13646 (2.10 s/question avg, ETA 1.9 min, failed 0, missing images 0)
done -- main, failed 0, missing images 0, 26.5 min


In [8]:
reef_ids = [qid for qid in all_sel if qid.startswith("reefknot_")]
reef_images = sorted({question_by_id[qid]["image_id"] for qid in reef_ids})
print(len(reef_ids), "reefknot questions on", len(reef_images), "images")
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    got = list(pool.map(find_vg_image_path, reef_images))
n_missing = sum(g is None for g in got)
print(f"fetched {len(reef_images) - n_missing} images in {time.time() - t0:.0f} s, missing {n_missing}")
assert n_missing == 0, "some Visual Genome images could not be fetched; rerun this cell"

2173 reefknot questions on 1628 images
fetched 1628 images in 63 s, missing 0


In [9]:
missing = collect("reefknot", lambda qid: qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

reefknot: 2173 questions, 0 already done, 2173 to run
[101/2173] reefknot_2387854_1 (1.40 s/question avg, ETA 48.4 min, failed 0, missing images 0)
[201/2173] reefknot_2410592_1 (1.46 s/question avg, ETA 47.9 min, failed 0, missing images 0)
[301/2173] reefknot_2337116_1 (1.44 s/question avg, ETA 45.1 min, failed 0, missing images 0)
[401/2173] reefknot_2399599_0 (1.43 s/question avg, ETA 42.3 min, failed 0, missing images 0)
[501/2173] reefknot_2416512_1 (1.44 s/question avg, ETA 40.0 min, failed 0, missing images 0)
[601/2173] reefknot_2411901_1 (1.45 s/question avg, ETA 38.0 min, failed 0, missing images 0)
[701/2173] reefknot_2316063_1 (1.46 s/question avg, ETA 35.8 min, failed 0, missing images 0)
[801/2173] reefknot_2375700_1 (1.45 s/question avg, ETA 33.2 min, failed 0, missing images 0)
[901/2173] reefknot_2373991_0 (1.45 s/question avg, ETA 30.8 min, failed 0, missing images 0)
[1001/2173] reefknot_2374793_0 (1.45 s/question avg, ETA 28.4 min, failed 0, missing images 0)
[1101

In [10]:
entries = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "*_batch_*.pt"))):
    entries.extend(torch.load(path))
ids = [e["question_id"] for e in entries]
print(f"{len(entries)} entries, {len(set(ids))} unique, {len(set(all_sel) - set(ids))} selected questions missing")
assert set(ids) == set(all_sel) and len(ids) == len(set(ids)), "collection incomplete or duplicated"

good = [e for e in entries if e["ok"]]
bad = [e for e in entries if not e["ok"]]
print(f"ok {len(good)}, failed {len(bad)}", {r: sum(e['reason'] == r for e in bad) for r in {e['reason'] for e in bad}})
print("noise level used:", {n: sum(e["noise_used"] == n for e in good) for n in NOISE_LEVELS})
print("gradient scale used:", {s: sum(e["grad_scale_used"] == s for e in good) for s in GRAD_SCALES})

dla_ld = {}
id_set = set(ids)
for path in glob.glob(f"{INPUT}/dla-qwen/*_batch_*.pt"):
    for e in torch.load(path):
        if e["question_id"] in id_set:
            dla_ld[e["question_id"]] = float(e["logit_diff"])
err = max(abs(e["sign"] * e["delta_clean"] - dla_ld[e["question_id"]]) for e in good)
print(f"clean logit difference vs dla-qwen over {len(good)} questions: max err {err:.4f}")
assert err < 0.1, "clean runs differ from the DLA collection"
assert len(bad) <= 0.02 * len(entries), "too many failed questions"

for stratum in ("false_yes", "true_yes", "true_no"):
    sub = [e for e in good if e["stratum"] == stratum]
    te = np.array([e["TE"] for e in sub])
    blind = np.array([e["sign"] * e["delta_corrupt"] for e in sub])
    print(f"{stratum:10s} n {len(sub):5d} | mean TE {te.mean():+.2f} | share with |TE| > 1: {np.mean(np.abs(te) > 1):.2f} | "
          f"mean delta clean {np.mean([e['delta_clean'] for e in sub]):+.2f} | blind Yes minus No {blind.mean():+.2f}")
print("sequence length: min", min(e["T"] for e in good), "max", max(e["T"] for e in good))
print("collection verified")

2927 entries, 2927 unique, 0 selected questions missing
ok 2927, failed 0 {}
noise level used: {3.0: 2927}
gradient scale used: {32.0: 2927, 1.0: 0}
clean logit difference vs dla-qwen over 2927 questions: max err 0.0312
false_yes  n  1119 | mean TE -3.14 | share with |TE| > 1: 0.83 | mean delta clean -2.90 | blind Yes minus No -0.25
true_yes   n  1085 | mean TE +4.47 | share with |TE| > 1: 0.91 | mean delta clean +3.89 | blind Yes minus No -0.58
true_no    n   723 | mean TE +2.22 | share with |TE| > 1: 0.74 | mean delta clean +6.85 | blind Yes minus No -4.63
sequence length: min 287 max 1051
collection verified


In [11]:
with open(os.path.join(OUT_DIR, "atp_meta.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "pairs": PAIRS, "n_layer": n_layer, "n_head": n_head, "groups": GROUPS,
        "tail_n": TAIL_N, "noise_levels": NOISE_LEVELS, "grad_scales": GRAD_SCALES, "direction": "noising",
        "n_ok": len(good), "n_failed": len(bad),
    }, f)

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "atp-qwen-noising",
        "id": f"{USERNAME}/atp-qwen-noising",
        "subtitle": "Noising attribution patching effects for Qwen2.5-VL yes/no questions",
        "description": "Per-question attribution patching for Qwen2.5-VL-7B on the 2927 questions in atp-qwen-selection. "
                       "Corrupt run: Gaussian noise added to the visual embeddings at layer 0 with std noise_used times the std of those embeddings (3.0). "
                       "The blind model leans strongly to No under this noise, so TE is large and negative for almost every hallucinated Yes. "
                       "Metric delta is the sign-corrected Yes minus No logit gap at the last position (capitalised pair for amber, lowercase pair for reefknot), "
                       "positive when the model favours the true answer. "
                       "head [3, 28, 28], mlp [3, 28] and rp [3, 28] hold the first-order effect of replacing each component's clean activation with its corrupt activation, "
                       "(corrupt minus clean) times the gradient of delta at the clean run, in logit units. The first axis is the position group: visual, text, tail (last 5 positions). "
                       "head is the attention head output before o_proj, mlp the MLP output, rp the residual stream entering the layer. "
                       "TE is delta_clean minus delta_corrupt. T and n_visual give the sequence length and image token count. Entries with ok false have no effects.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {OUT_DIR}

Starting upload for file reefknot_batch_10.pt
100%|██████████████████████████████████████| 1.84M/1.84M [00:00<00:00, 4.49MB/s]
Upload successful: reefknot_batch_10.pt (2MB)
Starting upload for file reefknot_batch_8.pt
100%|██████████████████████████████████████| 2.13M/2.13M [00:00<00:00, 4.81MB/s]
Upload successful: reefknot_batch_8.pt (2MB)
Starting upload for file reefknot_batch_1.pt
100%|██████████████████████████████████████| 2.13M/2.13M [00:00<00:00, 4.94MB/s]
Upload successful: reefknot_batch_1.pt (2MB)
Starting upload for file main_batch_0.pt
100%|██████████████████████████████████████| 2.12M/2.12M [00:00<00:00, 4.59MB/s]
Upload successful: main_batch_0.pt (2MB)
Starting upload for file reefknot_batch_4.pt
100%|██████████████████████████████████████| 2.13M/2.13M [00:00<00:00, 4.40MB/s]
Upload successful: reefknot_batch_4.pt (2MB)
Starting upload for file reefknot_batch_7.pt
100%|██████████████████████████████████████| 2.13M/2.13M [00:00<00:00, 4.67MB/s]
Upload successful: reefkn